# Data Preparation

In [0]:
# Create schema to RetailCast data
spark.sql("""
CREATE SCHEMA IF NOT EXISTS retailcast_solo.retailcast_solo_silver
""")

DataFrame[]

## Tabela "Informações de Loja"

In [0]:
from pyspark.sql.functions import col, to_date, regexp_replace, trim, date_format

# ==================================================
# (1)  INFO LOJA
# ==================================================

# Register the DataFrame as a temporary view so we can run SQL queries
# Example SQL query on the temporary view
result_df_raw = spark.sql('SELECT * FROM retailcast_solo.default.rc_info_loja')
result_df_raw.show()


# Data Cleansing and Transformation
df_silver = result_df_raw.withColumnRenamed("CAIXAS TRADICIONAIS", "CAIXAS_TRADICIONAIS") \
                         .withColumnRenamed("SELF CHECKOUTS", "SELF_CHECKOUTS") \
                         .withColumn("SECAO", regexp_replace(col("SECAO"), "Frente de loja", "Frente de Loja")) \
                        .withColumn("LOJA", trim(col("LOJA"))) \
                        .withColumn("ABERTURA", date_format(col("ABERTURA"), "HH:mm")) \
                        .withColumn("FECHO", date_format(col("FECHO"), "HH:mm"))

### RESUMO DAS ALTERAÇÕES:
# 1) Renomear e correção de nomes de lojas.
# 2) Normalização dos nomes de lojas.
# 3) A coluna ABERTURA e FECHO teve o formato da hora alterado para o formato HH:mm.

# Write the transformed data to the Silver layer
silver_data_path = "retailcast_solo.retailcast_solo_silver.rc_info_loja_silver"
df_silver.write.format("delta").mode("overwrite").saveAsTable(silver_data_path)

# Register the DataFrame as a temporary view so we can run SQL queries
df_silver.createOrReplaceTempView("rc_info_loja_silver")

# Example SQL query on the temporary view
result_df = spark.sql("SELECT COUNT(*) as trip_count FROM rc_info_loja_silver")

# Show the result of the query
result_df.show()

print("Silver layer processing completed.")

# AI Assisted Code

+-------+----------------+---------+--------------------+---------+--------------+--------+--------------+------------------+---------------+-----+-------------------+-------------------+-------------------+--------------+
|FK_LOJA|            LOJA|FK_CIDADE|              CIDADE|FK_REGIAO|        REGIAO|FK_SECAO|         SECAO|PRODUTIVIDADE/HORA|N_COLABORADORES| SKUS|           ABERTURA|              FECHO|CAIXAS_TRADICIONAIS|SELF_CHECKOUTS|
+-------+----------------+---------+--------------------+---------+--------------+--------+--------------+------------------+---------------+-----+-------------------+-------------------+-------------------+--------------+
| POR025|       Alfragide|      228|           Alfragide|       14|Lisboa Central|     976|Frente de Loja|               269|             85|35052|2000-01-01 08:00:00|2000-01-01 21:00:00|                 26|             8|
| POR012|    Almada Fórum|      164|              Almada|        3|    Margem Sul|      24|Frente de Loja|  

## Tabela "Feriados"

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, to_date, regexp_replace, trim, date_format, when, initcap, month, year, row_number, weekofyear

# ==================================================
# (2)  FERIADOS
# ==================================================

# Register the DataFrame as a temporary view so we can run SQL queries
# Example SQL query on the temporary view
result_df_raw = spark.sql('SELECT * FROM retailcast_solo.default.rc_feriados')
result_df_raw.show()

# Data Cleansing and Transformation
df_silver = result_df_raw.filter(col("FK_LOJA") != "POR454") \
    .withColumn("MES", month(col("DATA"))) \
    .withColumn("ANO", year(col("DATA"))) \
    .withColumn("SEMANA_DO_ANO", weekofyear(col("DATA"))) \
    .withColumn("DESCRICAO", trim(col("DESCRICAO"))) \
    .withColumn("ESTACAO_DO_ANO",
        when(col("MES").isin(3, 4, 5), "Primavera") \
        .when(col("MES").isin(6, 7, 8), "Verão") \
        .when(col("MES").isin(9, 10, 11), "Outono") \
        .otherwise("Inverno")
    ) \
.withColumn("TIPO",
        when(col("TIPO") == "ABERTO", 1)
        .when(col("TIPO") == "FECHADO", 0)
        .otherwise(None)
    ) \
    .withColumnRenamed("TIPO", "LOJA_ABERTA") \
    .withColumn("DESCRICAO",
        when(col("DESCRICAO") == "Dia da Liberedade", "Dia da Liberdade")
        .when(col("DESCRICAO").isin("Dia Ano Novo", "Dia de Ano Novo", "Passagem de Ano", "Passagem de ano", "passagem de ano"), "Ano Novo")
        .when(col("DESCRICAO").isin("Dia de natal", "Natal 2023"), "Natal")
        .when(col("DESCRICAO").isin("Sexta -Feira Santa", "Sexta-feira Santa"), "Sexta-Feira Santa")
        .when(col("DESCRICAO") == "Pascoa", "Páscoa")
        .when(col("DESCRICAO") == "Municipal (29 Jun)", "Feriado Municipal de Évora")
        .when(col("DESCRICAO").isin("Todos os Santos", "Feriado Dia de todos os Santos"), "Dia de Todos os Santos")
        .when(col("DESCRICAO") == "Imaculada Conceição", "Dia da Imaculada Conceição")
        .when(col("DESCRICAO") == "Feriado-carnaval", "Carnaval")
        .when(col("DESCRICAO") == "Feriado Municipa Vila Franca de Xira", "Feriado Municipal de Vila Franca de Xira")
        .when(col("DESCRICAO") == "Feriado Cidade Aveiro", "Feriado Municipal de Aveiro")
        .when(col("DESCRICAO") == "Feriado Cidade Viseu", "Feriado Municipal de Viseu")
        .when(col("DESCRICAO") == "Municipal Barreiro (28 Jun)", "Feriado Municipal do Barreiro")
        .when(col("DESCRICAO") == "1º MAIO - FERIADO ABERTO", "Dia do Trabalhador")
        .when(col("DESCRICAO") == "Dia de Bocage (municipal)", "Dia de Bocage")
        .when(col("DESCRICAO") == "Municipal de Setubal (15/09)", "Feriado Municipal de Setúbal")
        .when(col("DESCRICAO") == "Municipal do Porto (24 Jun)", "Feriado Municipal do Porto")
        .otherwise(col("DESCRICAO"))
    ) \
    .withColumn("DESCRICAO",
        when(col("LOJA") == "S - Ajuda", "Feriado Municipal de Ajuda")
        .when(col("LOJA") == "Vila do Conde", "Feriado Municipal de Vila do Conde")
        .when(col("LOJA") == "S - Cacilhas", "Feriado Municipal de Cacilhas")
        .when(col("LOJA") == "Torres Vedras", "Feriado Municipal de Torres Vedras")
        .when(col("LOJA") == "S - Costa Caparica (Pescadores)", "Feriado Municipal de Costa Caparica")
        .otherwise(col("DESCRICAO"))
     ) \
    .withColumn("DESCRICAO", initcap(col("DESCRICAO"))) \
                     .withColumn("DESCRICAO", regexp_replace(col("DESCRICAO"), r"\bDe\b", "de")) \
                     .withColumn("DESCRICAO", regexp_replace(col("DESCRICAO"), r"\bDa\b", "da")) \
                     .withColumn("DESCRICAO", regexp_replace(col("DESCRICAO"), r"\bOs\b", "os")) \
                     .withColumn("DESCRICAO", regexp_replace(col("DESCRICAO"), r"\bDo\b", "do"))


# ================================ FEATURES PARA ONE HOT ENCODING ================================================================

# window_spec = Window.partitionBy("FK_LOJA", "DESCRICAO").orderBy(col("ANO").desc())
# df_silver = df_silver.withColumn("row_num", row_number().over(window_spec)) \
#     .withColumn("REGISTO_MAIS_RECENTE", when(col("row_num") == 1, 1).otherwise(0)) \
#     .withColumn("VERAO", when(col("ESTACAO_DO_ANO") == "Verão", 1).otherwise(0)) \
#     .withColumn("PRIMAVERA", when(col("ESTACAO_DO_ANO") == "Primavera", 1).otherwise(0)) \
#     .withColumn("OUTONO", when(col("ESTACAO_DO_ANO") == "Outono", 1).otherwise(0)) \
#     .withColumn("INVERNO", when(col("ESTACAO_DO_ANO") == "Inverno", 1).otherwise(0))

# Obs.: valores = 1 = sim  // 0 = nao
# ================================================================================================================================

### RESUMO DAS ALTERAÇÕES:
# 1) Tabela Normalizada (nome dos feriados).
# 2) Adicionada colunas
#   . MES, ANO, SEMANA DO ANO, EPOCA DO ANO
#   . Colunas do Rich, com Hot Encoding, para indicar se o feriado é um dos mais importantes (Ano Novo, Carnaval, Natal, etc)
#   . Coluna TIPO alterarda para LOJA_ABERTA (este é o campo que indica se a loja estava aberta ou fechada no dia do feriado); Aplicado Hot Encoding.
#   . Adicionadas colunas HotEncoding pra estações do ano (VERAO, INVERNO, PRIMAVERA, OUTONO)
#   . REGISTO_RECENTE (este último é para nao perder dados e permitir filtrar no futuro).s
#   . A coluna REGISTO_RECENTE tem o valor = 1 quando é o mais recente feriado registado para aquela loja.
# 3) Removida a loja POR454 e junto a isso os valores nulos da coluna LOJA.

## ATENÇÂO:
# 1) Algumas lojas não o registo de todos os feriados nacionais (ano novo, carnaval), como:
#   POR025 (Alfragide): Falta-lhe o Ano Novo. Tem histórico de 2020 a 2024.
#   POR445 (S - Conde Redondo): Faltam-lhe o Dia da Liberdade e o Corpo de Deus. Tem histórico de 2020 a 2024.
#   U0143 (S - Ajuda): Falta-lhe a Sexta-Feira Santa. Tem histórico de 2020 a 2024.

# Write the transformed data to the Silver layer
silver_data_path = "retailcast_solo.retailcast_solo_silver.rc_feriados_silver"
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_data_path)

# Register the DataFrame as a temporary view so we can run SQL queries
df_silver.createOrReplaceTempView("rc_feriados_silver")

# Example SQL query on the temporary view
result_df = spark.sql("SELECT COUNT(*) as trip_count FROM rc_feriados_silver")

# Show the result of the query
result_df.show()

# AI Assisted Code

+-------+--------------------+----------+--------------------+------------+------+
|FK_LOJA|                LOJA|      DATA|           DESCRICAO|FERIADO_FIXO|  TIPO|
+-------+--------------------+----------+--------------------+------------+------+
| POR012|        Almada Fórum|2021-06-03|       Corpo de Deus|           N|ABERTO|
| POR417|S - Nuno Alvares ...|2021-06-03|       Corpo de Deus|           N|ABERTO|
| POR019|               Coina|2021-06-03|       Corpo de Deus|           N|ABERTO|
| POR012|        Almada Fórum|2021-06-10|     Dia de Portugal|           N|ABERTO|
| POR417|S - Nuno Alvares ...|2021-06-10|     Dia de Portugal|           N|ABERTO|
| POR019|               Coina|2021-06-10|     Dia de Portugal|           N|ABERTO|
| POR012|        Almada Fórum|2021-06-24|Municipal do Port...|           N|ABERTO|
| POR417|S - Nuno Alvares ...|2021-06-24|Municipal do Port...|           N|ABERTO|
| POR019|               Coina|2021-06-28|Municipal Barreir...|           N|ABERTO|
| PO

## Tabela "Eventos"

In [0]:
from pyspark.sql.functions import col, to_date, regexp_replace, trim, date_format, when, month

# ==================================================
# (3) EVENTOS
# ==================================================

# Register the DataFrame as a temporary view so we can run SQL queries
# Example SQL query on the temporary view
result_df_raw = spark.sql('SELECT * FROM retailcast_solo.default.rc_eventos')
result_df_raw.show()

# Data Cleansing and Transformation
df_silver = result_df_raw \
    .withColumn("DESCRICAO", trim(col("DESCRICAO"))) \
    .withColumn("MES_EVENTO",
        when(month(col("DATA_INI")) == month(col("DATA_FIM")), month(col("DATA_INI")))
        .otherwise(None)
    ) \
    .withColumn("ESTACAO_DO_ANO",
        when(col("MES_EVENTO").isin(3, 4, 5), "Primavera") \
        .when(col("MES_EVENTO").isin(6, 7, 8), "Verão") \
        .when(col("MES_EVENTO").isin(9, 10, 11), "Outono") \
        .otherwise("Inverno")
    ) \
    .withColumn("DESCRICAO",
        when(col("DESCRICAO") == "Folheto", "Folheto")
        .when(col("DESCRICAO").isin("Véspera Natal", "vespera natal"), "Véspera de Natal")
        .when(col("DESCRICAO") == "Véspera Ano Novo", "Véspera de Ano Novo")
        .when(col("DESCRICAO").isin("Inventário Cápsulas", "inventario capsulas cafe"), "Inventário Cápsulas Café")
        .when(col("DESCRICAO").isin("Inventário Garrafeira", "inventario garrafeira"), "Inventário Garrafeira")
        .when(col("DESCRICAO").isin("Inventário Lácteos", "inventario lacteos"), "Inventário Lácteos")
        .when(col("DESCRICAO") == "inventario ET", "Inventário ET")
        .when(col("DESCRICAO") == "inventario outdoor", "Inventário Outdoor")
        .when(col("DESCRICAO") == "Inventario NSBE", "Inventário NSBE")
        .when(col("DESCRICAO") == "INVENTARIO QUEIJOS", "Inventário Queijos")
        .when(col("DESCRICAO") == "invent", "Inventário")
        .when(col("DESCRICAO").isin("Ip Brico", "IP Brico"), "IP Brico")
        .when(col("DESCRICAO").isin("IP Artigos de mesa", "IP artigos de Mesa", "IP ARTIGOS MESA"), "IP Artigos de Mesa")
        .when(col("DESCRICAO").isin("IP Interiores", "IP Armazém Interiores"), "IP Interiores")
        .when(col("DESCRICAO").isin("IP Interiores armazém + brico"), "IP Interiores + Brico")
        .when(col("DESCRICAO").isin("IP PERFUMARIA", "IP Perfumaria", "IP PERFUMARIA ARMAZÉM"), "IP Perfumaria")
        .when(col("DESCRICAO").isin("IP Lácteos", "IP Lácteos/Congelados"), "IP Lácteos")
        .when(col("DESCRICAO") == "IP artigos de Mesa Armazém", "IP Artigos de Mesa Armazém")
        .when(col("DESCRICAO") == "IP  Peixaria", "IP Peixaria")
        .otherwise(col("DESCRICAO"))
    ) \

# ================================ FEATURES PARA ONE HOT ENCODING ================================================================

    # .withColumn("VERAO", when(col("ESTACAO_DO_ANO") == "Verão", 1).otherwise(0)) \
    # .withColumn("PRIMAVERA", when(col("ESTACAO_DO_ANO") == "Primavera", 1).otherwise(0)) \
    # .withColumn("OUTONO", when(col("ESTACAO_DO_ANO") == "Outono", 1).otherwise(0)) \
    # .withColumn("INVERNO", when(col("ESTACAO_DO_ANO") == "Inverno", 1).otherwise(0))

# Obs.: valores = 1 = sim  // 0 = nao
# ================================================================================================================================

### RESUMO DAS ALTERAÇÕES:
# 1) Duplicados removidos
# 2) Categorias de DESCRICAO normalizadas
# 3) Adicionada coluna "ESTACAO_DO_ANO".

# 1. Contar registos antes de remover duplicados
count_antes = df_silver.count()
print(f"Registos antes da remoção de duplicados exatos: {count_antes}")

# 2. Remover duplicados exatos e validar
df_silver = df_silver.dropDuplicates()

count_depois = df_silver.count()
print(f"Registos após a remoção: {count_depois}")
print(f"Total de duplicados eliminados: {count_antes - count_depois}")

# Write the transformed data to the Silver layer
silver_data_path = "retailcast_solo.retailcast_solo_silver.rc_eventos_silver"
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_data_path)

# Register the DataFrame as a temporary view so we can run SQL queries
df_silver.createOrReplaceTempView("rc_eventos_silver")

# Example SQL query on the temporary view
result_df = spark.sql("SELECT COUNT(*) as trip_count FROM rc_eventos_silver")

# Show the result of the query
result_df.show()

print("Silver layer processing completed.")

# AI Assisted Code

+-------+------------+----------+----------+--------------------+
|FK_LOJA|        LOJA|  DATA_INI|  DATA_FIM|           DESCRICAO|
+-------+------------+----------+----------+--------------------+
| POR012|Almada Fórum|2023-01-01|2023-01-01|       Janeiro 01.23|
| POR065|    Canidelo|2022-12-24|2022-12-24|       Véspera Natal|
| POR009|     Coimbra|2022-11-21|2022-11-21|       IP Garrafeira|
| POR009|     Coimbra|2022-11-14|2022-11-14|          IP Lácteos|
| POR062|       Eiras|2022-11-14|2022-11-14|Inventário Garraf...|
| POR062|       Eiras|2022-11-07|2022-11-07|    Entrada Campanha|
| POR062|       Eiras|2022-11-21|2022-11-21| Inventário Cápsulas|
| POR062|       Eiras|2022-11-09|2022-11-09|  Inventário Lácteos|
| POR008|        Gaia|2022-12-24|2022-12-24|       Véspera Natal|
| POR008|        Gaia|2022-12-31|2022-12-31|    Véspera Ano Novo|
| POR008|        Gaia|2022-12-24|2022-12-24|       Véspera Natal|
| POR008|        Gaia|2022-12-31|2022-12-31|    Véspera Ano Novo|
| POR008| 

## Tabela "Dados de venda"

In [0]:
from pyspark.sql.functions import col, to_date, regexp_replace, trim, date_format, when, month, year, weekofyear, dayofweek, dayofweek, mean, lit, dayofmonth

# ==================================================
# (4) DADOS VENDAS
# ==================================================

# Register the DataFrame as a temporary view so we can run SQL queries
# Example SQL query on the temporary view
result_df_raw = spark.sql('SELECT * FROM retailcast_solo.default.rc_dados_vendas')
result_df_raw.show()

## Calcular a média histórica da loja antes da data do outlier
media_hist_torres = result_df_raw.filter(
    (trim(col("LOJA")) == "Torres Vedras") &
    (month(col("DATA")) == 6) &
    (dayofmonth(col("DATA")) == 17) &
    (year(col("DATA")) != 2023) # Exclui apenas o ano do outlier, considerando anteriores e seguintes
).select(mean(col("VALOR"))).first()[0]

media_hist_arcos = result_df_raw.filter(
    (trim(col("LOJA")) == "Paço de Arcos") &
    (month(col("DATA")) == 12) &
    (dayofmonth(col("DATA")) == 23) &
    (year(col("DATA")) != 2021) # Exclui apenas o ano do outlier, considerando todos os outros
).select(mean(col("VALOR"))).first()[0]

# Data Cleansing and Transformation
df_silver = result_df_raw \
    .withColumn("LOJA", trim(col("LOJA"))) \
    .withColumn("MES", month(col("DATA"))) \
    .withColumn("ANO", year(col("DATA"))) \
    .withColumn("SEMANA_DO_ANO", weekofyear(col("DATA"))) \
    .withColumn("DIA_SEMANA", dayofweek(col("DATA"))) \
    .withColumn("VALOR",
        when(
            (col("LOJA") == "Torres Vedras") &
            (col("DATA") == "2023-06-17") &
            (col("VALOR") == 2134898.1483031157),
            lit(media_hist_torres)
        ).otherwise(col("VALOR"))
    ) \
    .withColumn("VALOR",
        when(
            (col("LOJA") == "Paço de Arcos") &
            (col("DATA") == "2021-12-23") &
            (col("VALOR") == 1253765.9217674492),
            lit(media_hist_arcos)
        ).otherwise(col("VALOR"))
    )


### RESUMO DAS ALTERAÇÕES:
# 1) Adicionadas colunas "LOJA", "MES", "ANO", "SEMANA_DO_ANO"
# 2) Outlier "Torres Vedras" corrigido e substituído pela média histórica da loja naquele dia


# Write the transformed data to the Silver layer
silver_data_path = "retailcast_solo.retailcast_solo_silver.rc_dados_vendas_silver"
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_data_path)

# Register the DataFrame as a temporary view so we can run SQL queries
df_silver.createOrReplaceTempView("rc_dados_vendas_silver")

# Example SQL query on the temporary view
result_df = spark.sql("SELECT COUNT(*) as trip_count FROM rc_dados_vendas_silver")

# Show the result of the query
result_df.show()

print("Silver layer processing completed.")

# AI Assisted Code

+--------+-----+----------+-----+------------------+------+------+--------------------+-------------------+
|FK_SECAO| LOJA|      DATA|ITENS|             VALOR|SKUS_+|SKUS_-|         VAR_PREÇO_+|        VAR_PREÇO_-|
+--------+-----+----------+-----+------------------+------+------+--------------------+-------------------+
|     522|Viseu|2020-01-05|33814|109862.21439241496|   891|  1581| 0.13838491982591622| 0.1275871252511443|
|     522|Viseu|2020-01-09|22926| 63819.58305887872|  1655|  1200| 0.10668822517201816| 0.1499412474641763|
|     522|Viseu|2020-01-24|26055| 72578.03103062364|   894|  1000| 0.12994644155209747|0.12363514335352539|
|     522|Viseu|2020-01-27|23556|  60853.0986604508|  1066|  1121| 0.18047729411634877|0.19849478032755793|
|     522|Viseu|2020-01-30|21653| 61461.64303757609|  1296|  1139| 0.16268606926069384|0.18100185512765804|
|     522|Viseu|2020-02-15|34480| 99093.91750765893|  1069|  1211| 0.11829461757923537| 0.1166891785097736|
|     522|Viseu|2020-02-21|2